# VinDr-Mammo Fast Downloader (aria2c)

Uses **aria2c** for maximum download speed with PhysioNet:
- **aria2c** handles server rate limiting better than wget
- **Batch downloads** to work around PhysioNet quotas
- **Connection reuse** for faster downloads
- **Smart retry** logic

## Why aria2c?
- wget parallel: PhysioNet queues requests → no speedup
- aria2c: Better connection management → actual speedup
- Handles authentication better for parallel downloads

---

In [ ]:
import os
import json
import subprocess
from pathlib import Path
from typing import List, Dict
import getpass
import pandas as pd
import numpy as np
from tqdm import tqdm
import time
from datetime import datetime

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

## Step 2: Install aria2c

In [ ]:
# Install aria2c (fast download tool)
!apt-get update -qq
!apt-get install -y -qq aria2
print("✅ aria2c installed")

## Step 3: Fast Downloader with aria2c

In [ ]:
class VinDrMammoFastDownloader:
    """
    Fast downloader using aria2c for better speed with PhysioNet.
    """
    
    def __init__(self, gdrive_path='/content/drive/MyDrive/vindr-mammo-stratified', 
                 max_total_files=1000, connections_per_file=5, concurrent_downloads=3):
        """
        Initialize fast downloader.
        
        Args:
            gdrive_path: Google Drive path
            max_total_files: Total files to download
            connections_per_file: Connections per file (aria2c -x)
            concurrent_downloads: Concurrent file downloads (aria2c -j)
        """
        self.base_dir = Path(gdrive_path)
        self.base_url = "https://physionet.org/files/vindr-mammo/1.0.0"
        
        self.username = None
        self.password = None
        
        self.max_total_files = max_total_files
        self.connections_per_file = connections_per_file
        self.concurrent_downloads = concurrent_downloads
        self.random_seed = 42
        
        # Create directories
        self.base_dir.mkdir(parents=True, exist_ok=True)
        (self.base_dir / 'images').mkdir(exist_ok=True)
        (self.base_dir / 'metadata').mkdir(exist_ok=True)
        
        self.progress_file = self.base_dir / 'download_progress.json'
        self.selection_file = self.base_dir / 'metadata' / 'selected_files.csv'
        
        print(f"✅ Initialized FAST downloader (aria2c) at {self.base_dir}")
        print(f"   Target: ~{max_total_files} files")
        print(f"   aria2c settings: {connections_per_file} connections/file, {concurrent_downloads} concurrent downloads")
    
    def setup_credentials(self, username: str = None, password: str = None) -> bool:
        """Setup credentials."""
        if not username:
            print("\n🔐 PhysioNet Credentials")
            username = input("Username: ").strip()
            password = getpass.getpass("Password: ")
        
        self.username = username
        self.password = password
        print("✅ Credentials saved")
        return True
    
    def download_metadata(self) -> bool:
        """Download metadata."""
        print("\n📊 Downloading Metadata")
        print("=" * 70)
        
        metadata_dir = self.base_dir / 'metadata'
        csv_file = 'breast-level_annotations.csv'
        url = f"{self.base_url}/{csv_file}"
        output_file = metadata_dir / csv_file
        
        if output_file.exists():
            print(f"  ✅ {csv_file} already exists")
            return True
        
        print(f"  📥 Downloading {csv_file}...")
        
        cmd = [
            'aria2c',
            f'--http-user={self.username}',
            f'--http-passwd={self.password}',
            '-d', str(metadata_dir),
            '-o', csv_file,
            '--max-tries=3',
            '--retry-wait=2',
            '-x', '5',  # 5 connections
            url
        ]
        
        try:
            result = subprocess.run(cmd, capture_output=True, timeout=90)
            
            if result.returncode == 0 and output_file.exists():
                size_mb = output_file.stat().st_size / (1024 * 1024)
                print(f"  ✅ Downloaded ({size_mb:.2f} MB)")
                return True
            else:
                print(f"  ❌ Failed: {result.stderr.decode()}")
                return False
        except Exception as e:
            print(f"  ❌ Error: {e}")
            return False
    
    def perform_stratified_selection(self) -> pd.DataFrame:
        """Perform stratified selection."""
        print("\n🎯 Performing Stratified Selection")
        print(f"   Target: ~{self.max_total_files} files")
        print("=" * 70)
        
        csv_file = self.base_dir / 'metadata' / 'breast-level_annotations.csv'
        
        if not csv_file.exists():
            print("❌ Metadata not found")
            return None
        
        df = pd.read_csv(csv_file)
        print(f"  Total images: {len(df)}")
        
        df['birads_numeric'] = df['breast_birads'].str.extract(r'(\d+)')[0].astype(float)
        df_filtered = df[df['birads_numeric'] != 3].copy()
        print(f"  After excluding BI-RADS 3: {len(df_filtered)}")
        
        df_filtered['label'] = df_filtered['birads_numeric'].apply(
            lambda x: 1 if x in [4, 5, 6] else 0
        )
        
        malignant_df = df_filtered[df_filtered['label'] == 1]
        benign_df = df_filtered[df_filtered['label'] == 0]
        
        print(f"\n  Classification:")
        print(f"    Malignant: {len(malignant_df)}")
        print(f"    Benign: {len(benign_df)}")
        
        # Patient-level sampling
        print(f"\n  Patient-level sampling...")
        np.random.seed(self.random_seed)
        
        target_malignant = int(self.max_total_files * 0.25)
        target_benign = int(self.max_total_files * 0.75)
        
        malignant_patient_sizes = malignant_df.groupby('study_id').size().reset_index(name='image_count')
        benign_patient_sizes = benign_df.groupby('study_id').size().reset_index(name='image_count')
        
        malignant_patient_sizes = malignant_patient_sizes.sample(frac=1, random_state=self.random_seed).reset_index(drop=True)
        benign_patient_sizes = benign_patient_sizes.sample(frac=1, random_state=self.random_seed).reset_index(drop=True)
        
        # Select patients
        selected_malignant_patients = []
        malignant_count = 0
        for _, row in malignant_patient_sizes.iterrows():
            if malignant_count >= target_malignant:
                break
            selected_malignant_patients.append(row['study_id'])
            malignant_count += row['image_count']
        
        selected_benign_patients = []
        benign_count = 0
        for _, row in benign_patient_sizes.iterrows():
            if benign_count >= target_benign:
                break
            selected_benign_patients.append(row['study_id'])
            benign_count += row['image_count']
        
        malignant_selected = malignant_df[malignant_df['study_id'].isin(selected_malignant_patients)]
        benign_selected = benign_df[benign_df['study_id'].isin(selected_benign_patients)]
        selected_df = pd.concat([malignant_selected, benign_selected], ignore_index=True)
        
        print(f"\n  ✅ Selection Complete:")
        print(f"    Malignant: {len(malignant_selected)} images from {len(selected_malignant_patients)} patients")
        print(f"    Benign: {len(benign_selected)} images from {len(selected_benign_patients)} patients")
        print(f"    Total: {len(selected_df)} images")
        
        selected_df.to_csv(self.selection_file, index=False)
        print(f"\n  💾 Saved to: {self.selection_file}")
        
        return selected_df
    
    def download_selected_files_aria2c(self, selected_df: pd.DataFrame = None, batch_size=50) -> bool:
        """
        Download using aria2c with batch processing.
        
        Args:
            selected_df: DataFrame with files
            batch_size: Files per batch (smaller = more progress updates)
        """
        print(f"\n📥 Downloading with aria2c (FAST MODE)")
        print("=" * 70)
        
        if selected_df is None:
            if not self.selection_file.exists():
                print("❌ No selection file")
                return False
            selected_df = pd.read_csv(self.selection_file)
        
        print(f"  Files to download: {len(selected_df)}")
        print(f"  Batch size: {batch_size}")
        print(f"  Concurrent downloads: {self.concurrent_downloads}")
        print(f"  Connections per file: {self.connections_per_file}\n")
        
        # Load progress
        progress = self._load_progress()
        downloaded_files = set(progress.get('downloaded_files', []))
        
        # Prepare download list
        files_to_download = []
        for idx, row in selected_df.iterrows():
            study_id = row['study_id']
            image_id = row['image_id']
            image_path = f"images/{study_id}/{image_id}.dicom"
            
            if image_path not in downloaded_files:
                output_file = self.base_dir / image_path
                output_file.parent.mkdir(parents=True, exist_ok=True)
                
                url = f"{self.base_url}/{image_path}"
                files_to_download.append((url, output_file, image_path))
        
        if not files_to_download:
            print("  ✅ All files already downloaded!")
            return True
        
        print(f"  📥 Need to download: {len(files_to_download)} files\n")
        
        # Download in batches
        total_success = 0
        total_failed = 0
        
        num_batches = (len(files_to_download) + batch_size - 1) // batch_size
        
        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(files_to_download))
            batch = files_to_download[start_idx:end_idx]
            
            print(f"  Batch {batch_idx + 1}/{num_batches}: Downloading {len(batch)} files...")
            
            # Create aria2c input file
            input_file = self.base_dir / f'aria2c_batch_{batch_idx}.txt'
            
            with open(input_file, 'w') as f:
                for url, output_file, _ in batch:
                    f.write(f"{url}\n")
                    f.write(f"  out={output_file}\n")
                    f.write(f"  http-user={self.username}\n")
                    f.write(f"  http-passwd={self.password}\n")
            
            # Run aria2c
            cmd = [
                'aria2c',
                '-i', str(input_file),
                f'-j{self.concurrent_downloads}',  # Concurrent downloads
                f'-x{self.connections_per_file}',  # Connections per file
                '--max-tries=3',
                '--retry-wait=2',
                '--timeout=60',
                '--allow-overwrite=true',
                '--auto-file-renaming=false',
                '--summary-interval=5'
            ]
            
            try:
                result = subprocess.run(cmd, capture_output=False, timeout=batch_size * 60)
                
                # Check which files downloaded
                batch_success = 0
                batch_failed = 0
                
                for url, output_file, image_path in batch:
                    if output_file.exists() and output_file.stat().st_size > 0:
                        batch_success += 1
                        progress['downloaded_files'].append(image_path)
                    else:
                        batch_failed += 1
                        progress.setdefault('failed_files', []).append(image_path)
                
                total_success += batch_success
                total_failed += batch_failed
                
                print(f"    ✅ Batch complete: {batch_success} success, {batch_failed} failed")
                
                # Clean up input file
                input_file.unlink()
                
                # Save progress
                self._save_progress(progress)
                
            except Exception as e:
                print(f"    ❌ Batch failed: {e}")
                input_file.unlink()
        
        print(f"\n{'=' * 70}")
        print(f"✅ Download Complete!")
        print(f"   Success: {total_success}")
        print(f"   Failed: {total_failed}")
        print(f"   Total: {len(progress['downloaded_files'])}")
        print(f"{'=' * 70}\n")
        
        return total_failed == 0
    
    def _load_progress(self) -> Dict:
        """Load progress."""
        if self.progress_file.exists():
            try:
                with open(self.progress_file, 'r') as f:
                    return json.load(f)
            except:
                pass
        return {'downloaded_files': [], 'failed_files': []}
    
    def _save_progress(self, progress: Dict):
        """Save progress."""
        progress['last_update'] = datetime.now().isoformat()
        with open(self.progress_file, 'w') as f:
            json.dump(progress, f, indent=2)

## Step 4: Initialize and Run

In [ ]:
# Initialize downloader
downloader = VinDrMammoFastDownloader(
    gdrive_path='/content/drive/MyDrive/vindr-mammo-stratified',
    max_total_files=1000,
    connections_per_file=5,     # More connections = faster (but not too many)
    concurrent_downloads=3      # Download 3 files at once (safe for PhysioNet)
)

In [ ]:
# Setup credentials
downloader.setup_credentials()

In [ ]:
# Download metadata
downloader.download_metadata()

In [ ]:
# Perform selection
selected_df = downloader.perform_stratified_selection()

In [ ]:
# Download with aria2c (FAST!)
downloader.download_selected_files_aria2c(
    selected_df,
    batch_size=50  # Download in batches of 50
)

## Troubleshooting: If Still Slow

If downloads are still slow, PhysioNet likely has hard rate limits. Try:

1. **Reduce concurrent downloads:**
   ```python
   concurrent_downloads=1  # Sequential, but more reliable
   connections_per_file=10 # Use more connections per file instead
   ```

2. **Check PhysioNet status:**
   - Server might be slow/overloaded
   - Check https://physionet.org/ for announcements

3. **Download overnight:**
   - Even at 35 sec/file, it will complete
   - Colab stays active if window is open

---